# Task_3_Build
## Workflow Graph


In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END

class SupportState(TypedDict):
    query: str
    intent: str
    response: str

def classify(state):
    q = state['query'].lower()
    state['intent'] = ('billing' if any(w in q for w in ['bill','refund'])
        else 'technical' if any(w in q for w in ['error','crash'])
        else 'escalation' if 'human' in q else 'faq')
    return state

def billing(state):
    state['response'] = 'Billing: Refunds processed in 3-5 days.'; return state
def technical(state):
    state['response'] = 'Technical: Please clear cache and retry.'; return state
def faq(state):
    state['response'] = 'FAQ: Check our help docs at help.techcorp.com'; return state
def escalation(state):
    state['response'] = 'Connecting you to a human agent now...'; return state

def router(state) -> Literal['billing','technical','faq','escalation']:
    return state['intent']

graph = StateGraph(SupportState)
for name, fn in [('classify',classify),('billing',billing),('technical',technical),('faq',faq),('escalation',escalation)]:
    graph.add_node(name, fn)
graph.set_entry_point('classify')
graph.add_conditional_edges('classify', router)
for n in ['billing','technical','faq','escalation']: graph.add_edge(n, END)

app = graph.compile()

test_queries = ['My bill is wrong','App keeps crashing','I want to talk to a human','What are your hours?']
for q in test_queries:
    result = app.invoke({'query': q, 'intent': '', 'response': ''})
    print(f'Q: {q}\n  -> [{result["intent"]}] {result["response"]}\n')